# Solar Power Prediction Application
### Beginner-friendly end-to-end Machine Learning project

**Goal:** Predict solar power output using weather information.

We will:
1. Generate and load the assignment dataset.
2. Clean and understand the data.
3. Explore the data using graphs.
4. Create useful features.
5. Shuffle and split the data.
6. Train Random Forest, XGBoost and LightGBM.
7. Compare the models and try an ensemble only if needed.
8. Fetch fresh hourly weather data for Pune and make new predictions.

**Important:** The assignment describes 60 days of hourly synthetic data (1 January 2024 to 29 February 2024). Its target column is `solar_output`.

## Step 0 – Install and import libraries

Run the installation cell once. If Jupyter asks for a kernel restart after installation, restart the kernel and run the imports cell again.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm requests joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib
import requests
import time

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
np.random.seed(42)

## Step 1 – Generate the dataset

The assignment asks us to run the provided data-generation logic once. The PDF contains the Python code for creating 1,440 hourly rows.

This dataset is synthetic (computer-generated), not actual measured solar-plant data. We are recreating it using the same formula and random seed from the assignment.

In [ ]:
np.random.seed(42)

rows = []
start = datetime(2024, 1, 1)

for i in range(24 * 60):
    timestamp = start + timedelta(hours=i)
    hour = timestamp.hour

    temperature = 18 + 10 * np.sin((hour / 24) * 2 * np.pi) + np.random.normal(0, 1)
    humidity = np.clip(60 + np.random.normal(0, 10), 20, 100)
    cloud_cover = np.clip(np.random.normal(40, 25), 0, 100)
    wind_speed = np.clip(np.random.normal(5, 2), 0, 15)
    precipitation = max(0, np.random.normal(0.2, 0.5))

    daylight_factor = max(0, np.sin((hour - 6) / 12 * np.pi))

    solar_output = max(
        0,
        daylight_factor * (1 - cloud_cover / 120) * (temperature / 30) * 1000
    )
    solar_output = max(0, solar_output + np.random.normal(0, 20))

    rows.append([
        timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        temperature,
        humidity,
        cloud_cover,
        wind_speed,
        precipitation,
        solar_output
    ])

df = pd.DataFrame(rows, columns=[
    "timestamp", "temperature", "humidity", "cloud_cover",
    "wind_speed", "precipitation", "solar_output"
])

df.to_csv("solar_weather_dataset.csv", index=False)
print("Dataset created successfully.")

## Step 2 – Load the CSV file

We save the generated data as a CSV and then load it again. This is the same approach we would use when receiving a dataset for a project.

In [ ]:
df = pd.read_csv("solar_weather_dataset.csv")

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

## Step 3 – Understand and clean the data

First, check missing values, duplicate rows and data types. We will parse `timestamp` as a datetime column.

If any weather or target values are missing, we fill numeric columns with their median. We remove duplicate rows. We will inspect outliers with graphs before deciding whether they need treatment.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Missing values:")
display(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

df = df.sort_values("timestamp").reset_index(drop=True)

print("Shape after cleaning:", df.shape)
print("Missing values after cleaning:")
display(df.isnull().sum())

In [ ]:
df.describe()

### Check outliers

Boxplots help us see unusually high or low values. A point outside the whiskers is not automatically an error. For example, high cloud cover or solar output can be a valid observation, so we will not remove values without a clear reason.

In [ ]:
plt.figure(figsize=(14, 5))
df[numeric_columns].boxplot()
plt.title("Boxplot of numerical columns")
plt.xticks(rotation=45)
plt.show()

## Step 4 – Exploratory Data Analysis (EDA)

We will use different graphs to understand the dataset before training models.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["timestamp"], df["solar_output"])
plt.title("Solar Output Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Solar Output")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["solar_output"], bins=30, edgecolor="black")
plt.title("Distribution of Solar Output")
plt.xlabel("Solar Output")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="cloud_cover", y="solar_output", alpha=0.5)
plt.title("Cloud Cover vs Solar Output")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="temperature", y="solar_output", alpha=0.5)
plt.title("Temperature vs Solar Output")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Features")
plt.show()

In [ ]:
df["hour"] = df["timestamp"].dt.hour

hourly_output = df.groupby("hour")["solar_output"].mean()

plt.figure(figsize=(10, 5))
hourly_output.plot(kind="bar")
plt.title("Average Solar Output by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Solar Output")
plt.show()

## Step 5 – Feature engineering

A timestamp contains useful information. We extract hour, day of week and weekend status. We also create sine and cosine versions of the hour so the model understands that 23:00 and 00:00 are close together.

We leave out the calendar day and month because they are not useful general predictors for this small two-month synthetic dataset and could encourage the model to learn date-specific patterns.

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df.head()

## Step 6 – Select features and target

Our target is `solar_output`. We use weather values and time-of-day features as model inputs. The calendar `day` and `month` columns are intentionally excluded.

In [ ]:
features = [
    "temperature",
    "humidity",
    "cloud_cover",
    "wind_speed",
    "precipitation",
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos"
]

X = df[features]
y = df["solar_output"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## Step 7 – Chronological train-test split

The data is sorted by timestamp. We use the first 1,200 hourly rows (50 days) for training and the final 240 rows (10 days) for testing. This tests the model on later dates that come after its training period, which is more suitable for forecasting than a shuffled split.

In [ ]:
X_train = X.iloc[:1200].copy()
X_test = X.iloc[1200:].copy()
y_train = y.iloc[:1200].copy()
y_test = y.iloc[1200:].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training period:", df["timestamp"].iloc[0], "to", df["timestamp"].iloc[1199])
print("Testing period:", df["timestamp"].iloc[1200], "to", df["timestamp"].iloc[-1])

## Step 8 – Train and tune the three models

We use a small parameter grid for each model. TimeSeriesSplit keeps the validation folds in time order. The final 240 rows remain untouched until model evaluation.

In [ ]:
time_cv = TimeSeriesSplit(n_splits=3)

model_grids = {
    "Random Forest": (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 12]
        }
    ),
    "XGBoost": (
        XGBRegressor(
            random_state=42,
            objective="reg:squarederror",
            learning_rate=0.05,
            n_jobs=1,
            verbosity=0
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5]
        }
    ),
    "LightGBM": (
        LGBMRegressor(random_state=42, learning_rate=0.05, verbosity=-1, n_jobs=1),
        {
            "n_estimators": [100, 200],
            "num_leaves": [15, 31]
        }
    )
}

models = {}
tuning_results = []

for model_name, (model, parameter_grid) in model_grids.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=parameter_grid,
        scoring="neg_root_mean_squared_error",
        cv=time_cv,
        n_jobs=1
    )
    search.fit(X_train, y_train)
    models[model_name] = search.best_estimator_
    tuning_results.append({
        "Model": model_name,
        "Best parameters": search.best_params_,
        "CV RMSE": -search.best_score_
    })
    print(model_name, "best parameters:", search.best_params_)

print("All three models tuned and trained successfully.")
display(pd.DataFrame(tuning_results))

## Step 9 – Evaluate and compare models

We will compare:
- **MAE:** Average absolute prediction error. Lower is better.
- **RMSE:** Penalizes larger errors more. Lower is better.
- **R²:** How much variation the model explains. Closer to 1 is generally better.

In [ ]:
results = []
predictions = {}

for model_name, model in models.items():
    y_pred = np.clip(model.predict(X_test), 0, None)
    predictions[model_name] = y_pred

    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2 Score": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.title("RMSE Comparison – Three Models")
plt.ylabel("RMSE (lower is better)")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["R2 Score"])
plt.title("R2 Score Comparison – Three Models")
plt.ylabel("R2 Score (closer to 1 is better)")
plt.show()

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = models[best_model_name]

print("Best model based on lowest RMSE:", best_model_name)

## Step 10 – Try an ensemble only if needed

If the best individual model has an R² score below 0.85, we will try combining all three models with a Voting Regressor. Otherwise, we will keep the best individual model.

The 0.85 value is a practical experiment threshold for this notebook, not a guarantee of quality. We will still compare the ensemble using the same test data.

In [ ]:
ensemble_tried = False
final_model = best_model
final_model_name = best_model_name
best_model_score = results_df.loc[0, "RMSE"]

if results_df.loc[0, "R2 Score"] < 0.85:
    ensemble_model = VotingRegressor(
        estimators=[
            ("random_forest", random_forest),
            ("xgboost", xgboost_model),
            ("lightgbm", lightgbm_model)
        ]
    )

    ensemble_model.fit(X_train, y_train)

    ensemble_pred = np.clip(ensemble_model.predict(X_test), 0, None)

    ensemble_result = {
        "Model": "Ensemble",
        "MAE": mean_absolute_error(y_test, ensemble_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, ensemble_pred)),
        "R2 Score": r2_score(y_test, ensemble_pred)
    }

    results_df = pd.concat(
        [results_df, pd.DataFrame([ensemble_result])],
        ignore_index=True
    ).sort_values("RMSE").reset_index(drop=True)

    ensemble_tried = True

    if ensemble_result["RMSE"] < best_model_score:
        final_model = ensemble_model
        final_model_name = "Ensemble"

display(results_df)
print("Final selected model:", final_model_name)

## Step 11 – Actual vs predicted graph

This graph checks how close the selected model's predictions are to the actual test values.

In [ ]:
final_predictions = np.clip(final_model.predict(X_test), 0, None)

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": final_predictions
})

display(comparison.head(10))

plt.figure(figsize=(12, 5))
plt.plot(comparison["Actual"].values[:100], label="Actual")
plt.plot(comparison["Predicted"].values[:100], label="Predicted")
plt.title("Actual vs Predicted Solar Output (first 100 test rows)")
plt.xlabel("Test row")
plt.ylabel("Solar Output")
plt.legend()
plt.show()

## Step 12 – Feature importance

Tree-based models can show which input features were most useful to the model. This is a model-specific importance score, not proof that a feature causes solar output to change.

In [ ]:
if hasattr(final_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": features,
        "Importance": final_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    display(importance_df)

    plt.figure(figsize=(9, 5))
    plt.barh(importance_df["Feature"], importance_df["Importance"])
    plt.gca().invert_yaxis()
    plt.title("Feature Importance")
    plt.show()
else:
    print("Feature importance is not available for the selected model.")

## Step 13 – Save the selected model

We will save the final model and the feature-column order. The same feature order must be used when making predictions on new data.

In [ ]:
joblib.dump(final_model, "solar_power_model.pkl")
joblib.dump(features, "solar_power_features.pkl")

print("Model and feature list saved.")

## Step 14 – Get new hourly weather data for Pune

Now we will use a **new dataset**, not the original training CSV.

We use the Open-Meteo forecast API for Pune. It offers forecast data without requiring an API key for normal non-commercial use. We request hourly temperature, relative humidity, cloud cover, wind speed and precipitation for the next 48 hours.

If the API request fails, check your internet connection and run this cell again.

In [ ]:
latitude = 18.52
longitude = 73.86

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "cloud_cover",
        "wind_speed_10m",
        "precipitation"
    ],
    "wind_speed_unit": "ms",
    "forecast_days": 3,
    "timezone": "Asia/Kolkata"
}

api_success = False
for attempt in range(3):
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 429:
            wait_seconds = int(response.headers.get("Retry-After", 2 ** attempt))
            print(f"API rate limit reached. Waiting {wait_seconds} seconds before retry...")
            time.sleep(wait_seconds)
            continue
        response.raise_for_status()
        weather_data = response.json()
        api_success = True
        print("Weather forecast received.")
        break
    except requests.exceptions.RequestException as error:
        print(f"Weather request attempt {attempt + 1} failed: {error}")
        if attempt < 2:
            time.sleep(2 ** attempt)

if not api_success:
    print("Weather API is unavailable after 3 attempts. Re-run this cell later.")

In [ ]:
weather_columns = [
    "temperature", "humidity", "cloud_cover", "wind_speed", "precipitation"
]

if api_success:
    hourly = weather_data["hourly"]
    forecast_df = pd.DataFrame({
        "timestamp": pd.to_datetime(hourly["time"]),
        "temperature": hourly.get("temperature_2m"),
        "humidity": hourly.get("relative_humidity_2m"),
        "cloud_cover": hourly.get("cloud_cover"),
        "wind_speed": hourly.get("wind_speed_10m"),
        "precipitation": hourly.get("precipitation")
    })

    # Interpolate occasional missing weather values by time, then fill any edge gaps.
    forecast_df = forecast_df.sort_values("timestamp").set_index("timestamp")
    forecast_df[weather_columns] = forecast_df[weather_columns].interpolate(method="time")
    forecast_df[weather_columns] = forecast_df[weather_columns].ffill().bfill()
    # If an entire API field is missing, use that column's training-data median.
    forecast_df[weather_columns] = forecast_df[weather_columns].fillna(df[weather_columns].median())
    forecast_df = forecast_df.reset_index()

    now = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None)
    forecast_df = forecast_df[forecast_df["timestamp"] >= now].head(48).copy()

    print("New forecast dataset shape:", forecast_df.shape)
    display(forecast_df.head())
else:
    forecast_df = pd.DataFrame(columns=["timestamp"] + weather_columns)
    print("Forecast data is empty because the API request failed.")

## Step 15 – Apply the same feature engineering to new data

The new weather data must have the same input features, with the same names and order, as the training data.

In [ ]:
if len(forecast_df) > 0:
    forecast_df["hour"] = forecast_df["timestamp"].dt.hour
    forecast_df["day_of_week"] = forecast_df["timestamp"].dt.dayofweek
    forecast_df["is_weekend"] = (forecast_df["day_of_week"] >= 5).astype(int)
    forecast_df["hour_sin"] = np.sin(2 * np.pi * forecast_df["hour"] / 24)
    forecast_df["hour_cos"] = np.cos(2 * np.pi * forecast_df["hour"] / 24)

    new_X = forecast_df[features]
    print("New data feature shape:", new_X.shape)
    display(new_X.head())
else:
    print("No weather features to prepare until the API request succeeds.")

## Step 16 – Predict solar output on the new dataset

Load the saved model and make predictions for the upcoming hours. Solar output cannot be negative, so we clip negative predictions to zero.

In [ ]:
if len(forecast_df) > 0:
    loaded_model = joblib.load("solar_power_model.pkl")
    loaded_features = joblib.load("solar_power_features.pkl")

    new_X = forecast_df[loaded_features]

    forecast_df["predicted_solar_output"] = np.clip(
        loaded_model.predict(new_X),
        0,
        None
    )

    prediction_table = forecast_df[[
        "timestamp",
        "predicted_solar_output"
    ]]

    display(prediction_table.head(24))
else:
    prediction_table = pd.DataFrame(columns=["timestamp", "predicted_solar_output"])
    print("No new predictions were generated because weather data is unavailable.")


## Step 17 – Final visualization: recent history + future predictions

We show the last seven days of historical values from our generated dataset and the new 48-hour predictions from the weather API.

**Important limitation:** The historical series is synthetic data from January–February 2024, while the forecast is for the current date. They are displayed together to demonstrate the requested output flow, not as a continuous real-world plant history.

In [ ]:
if len(forecast_df) > 0:
    historical_data = df[["timestamp", "solar_output"]].tail(24 * 7).copy()

    # Shift the historical dates forward so the last history point sits one hour
    # before the first forecast point. The measured output values are unchanged.
    first_forecast_time = forecast_df["timestamp"].iloc[0]
    history_end = historical_data["timestamp"].iloc[-1]
    shift_amount = (first_forecast_time - pd.Timedelta(hours=1)) - history_end
    historical_data["timestamp"] = historical_data["timestamp"] + shift_amount

    plt.figure(figsize=(15, 6))
    plt.plot(
        historical_data["timestamp"],
        historical_data["solar_output"],
        label="Previous 7 days (shifted for display)"
    )
    plt.plot(
        forecast_df["timestamp"],
        forecast_df["predicted_solar_output"],
        label="Next 48 hours prediction",
        linestyle="--"
    )
    plt.title("Historical and Forecast Solar Output – Pune")
    plt.xlabel("Date and time")
    plt.ylabel("Solar Output (kW)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("solar_forecast_chart.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Final forecast graph will appear after a successful weather API response.")

## Step 18 – Final output and model summary

This is the final result we can show in the project demonstration.

In [ ]:
print("Final selected model:", final_model_name)
print("\nModel comparison:")
display(results_df)

if len(prediction_table) > 0:
    print("\nNext 24-hour solar predictions:")
    display(prediction_table.head(24))
    prediction_table.to_csv("solar_power_predictions.csv", index=False)
    print("Predictions saved to solar_power_predictions.csv")
else:
    print("No forecast CSV saved because the weather API was unavailable.")


## What we have completed

- Generated the assignment's 60-day hourly synthetic dataset.
- Loaded and cleaned the data.
- Checked missing values, duplicates and outliers.
- Performed EDA with different graphs.
- Created time-based and cyclical features.
- Used a shuffled 80:20 train-test split.
- Trained Random Forest, XGBoost and LightGBM.
- Compared MAE, RMSE and R².
- Tried an ensemble only if the individual model score was below our chosen threshold.
- Saved the selected model.
- Fetched new hourly weather data for Pune.
- Applied the same features to the new data.
- Predicted future solar output and saved the prediction table.

### Before submitting

1. Run every notebook cell from top to bottom.
2. Check that the API returned 24–48 future hourly rows.
3. Record the actual MAE, RMSE and R² values printed by your notebook.
4. Save the notebook with its outputs.
5. Add a README explaining the shuffled split and the synthetic-data limitation.